# 2 · Numbers and Arithmetic

*Language basics* · 10 exercises

Floor division, big integers, modular arithmetic and the float traps that cost contest points.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-2) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-2)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import math

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 2.1 · Floor division with negatives  `Core`

**Task.** Write `div_mod(a, b)` returning `(a // b, a % b)`. Predict the results for negative inputs before running it.

**Example.** `div_mod(7, 2)` -> `(3, 1)`; `div_mod(-7, 2)` -> `(-4, 1)`

**Big O.** O(1).

**Note.** Python floors toward minus infinity, so `-7 // 2` is -4 and `-7 % 2` is 1. C++ and Java truncate toward zero and give -3 and -1.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-2-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-2-1)

In [ ]:
# 2.1 Floor division with negatives
def div_mod(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_2_1():
    assert div_mod(7, 2) == (3, 1)
    assert div_mod(-7, 2) == (-4, 1)


test_ex_2_1()
print('2.1 passed')

### 2.2 · Ceiling division without floats  `Core`

**Task.** Write `ceil_div(a, b)` for integers using only `//` (no `math.ceil`, no `/`).

**Example.** `ceil_div(7, 2)` -> `4`; `ceil_div(6, 3)` -> `2`; `ceil_div(-7, 2)` -> `-3`

**Big O.** O(1).

**Note.** `math.ceil(a / b)` goes through a float and is wrong for large values such as `10**18 + 1`.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-2-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-2-2)

In [ ]:
# 2.2 Ceiling division without floats
def ceil_div(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_2_2():
    for a in range(-20, 21):
        for b in (1, 2, 3, 7, -3):
            assert ceil_div(a, b) == math.ceil(a / b)
    assert ceil_div(10 ** 18 + 1, 1) == 10 ** 18 + 1


test_ex_2_2()
print('2.2 passed')

### 2.3 · Big integers for free  `Core`

**Task.** Write `factorial_digits(n)` returning how many digits n! has. Python ints never overflow, so compute it directly.

**Example.** `factorial_digits(10)` -> `7` (3628800); `factorial_digits(100)` -> `158`

**Big O.** Fine for n up to a few thousand; the number itself has O(n log n) digits.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-2-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-2-3)

In [ ]:
# 2.3 Big integers for free
def factorial_digits(n):
    raise NotImplementedError

In [ ]:
def test_ex_2_3():
    assert factorial_digits(10) == 7
    assert factorial_digits(100) == 158


test_ex_2_3()
print('2.3 passed')

### 2.4 · Modular power and inverse  `Core`

**Task.** With `MOD = 10**9 + 7`, write `mod_pow(a, b)` using three-argument `pow` and `mod_inv(a)` using `pow(a, -1, MOD)`. Check that `a * mod_inv(a) % MOD == 1`.

**Example.** `mod_pow(2, 10)` -> `1024`; `mod_pow(2, 100)` -> `976371285`

**Big O.** O(log b) by repeated squaring.

**Note.** `pow(a, b) % MOD` computes the full huge number first; `pow(a, b, MOD)` never does.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-2-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-2-4)

In [ ]:
# 2.4 Modular power and inverse
MOD = 10**9 + 7


def mod_pow(a, b):
    raise NotImplementedError


def mod_inv(a):
    raise NotImplementedError

In [ ]:
def test_ex_2_4():
    assert mod_pow(2, 10) == 1024
    assert mod_pow(2, 100) == 976371285
    assert 123456 * mod_inv(123456) % MOD == 1


test_ex_2_4()
print('2.4 passed')

### 2.5 · Comparing floats safely  `Core`

**Task.** Write `close(a, b, eps=1e-9)` and show that `0.1 + 0.2 == 0.3` is False but `close(0.1 + 0.2, 0.3)` is True.

**Example.** `close(0.1 + 0.2, 0.3)` -> `True`

**Big O.** O(1).

**Note.** `math.isclose` exists too; contest problems usually state an absolute or relative tolerance, so match theirs.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-2-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-2-5)

In [ ]:
# 2.5 Comparing floats safely
def close(a, b, eps=1e-9):
    raise NotImplementedError

In [ ]:
def test_ex_2_5():
    assert 0.1 + 0.2 != 0.3
    assert close(0.1 + 0.2, 0.3)
    assert not close(0.1, 0.2)


test_ex_2_5()
print('2.5 passed')

### 2.6 · Perfect squares with isqrt  `Core`

**Task.** Write `is_square(n)` with `math.isqrt`. Then explain why `int(n ** 0.5) ** 2 == n` can fail for large n.

**Example.** `is_square(49)` -> `True`; `is_square(10**30 + 1)` -> `False`; `is_square((10**15 + 3) ** 2)` -> `True`

**Big O.** O(1) for contest-sized numbers.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-2-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-2-6)

In [ ]:
# 2.6 Perfect squares with isqrt
def is_square(n):
    raise NotImplementedError

In [ ]:
def test_ex_2_6():
    assert is_square(49)
    assert not is_square(10 ** 30 + 1)
    assert is_square((10 ** 15 + 3) ** 2)
    assert not is_square(-4)


test_ex_2_6()
print('2.6 passed')

### 2.7 · Digit sum two ways  `Core`

**Task.** Write `digit_sum_str(n)` using `str` and `digit_sum_math(n)` using `% 10` and `// 10`. Both work for negative n by ignoring the sign.

**Example.** `digit_sum_str(9045)` -> `18`; `digit_sum_math(-123)` -> `6`

**Big O.** O(D) for D digits.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-2-7) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-2-7)

In [ ]:
# 2.7 Digit sum two ways
def digit_sum_str(n):
    raise NotImplementedError


def digit_sum_math(n):
    raise NotImplementedError

In [ ]:
def test_ex_2_7():
    assert digit_sum_str(9045) == 18 == digit_sum_math(9045)
    assert digit_sum_math(-123) == 6


test_ex_2_7()
print('2.7 passed')

### 2.8 · Reverse the digits  `Core`

**Task.** Write `reverse_int(n)` keeping the sign and dropping leading zeros.

**Example.** `reverse_int(1200)` -> `21`; `reverse_int(-345)` -> `-543`

**Big O.** O(D).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-2-8) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-2-8)

In [ ]:
# 2.8 Reverse the digits
def reverse_int(n):
    raise NotImplementedError

In [ ]:
def test_ex_2_8():
    assert reverse_int(1200) == 21
    assert reverse_int(-345) == -543


test_ex_2_8()
print('2.8 passed')

### 2.9 · Number bases  `Core`

**Task.** Write `to_base(n, b)` returning the digits of non-negative n in base b (2 to 36) as a string, and use `int(s, b)` to convert back. Compare with `bin`, `oct`, `hex`.

**Example.** `to_base(255, 16)` -> `"FF"`; `to_base(10, 2)` -> `"1010"`; `int("1010", 2)` -> `10`

**Big O.** O(log_b n).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-2-9) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-2-9)

In [ ]:
# 2.9 Number bases
DIGITS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"


def to_base(n, b):
    raise NotImplementedError

In [ ]:
def test_ex_2_9():
    assert to_base(255, 16) == 'FF'
    assert to_base(10, 2) == '1010'
    assert to_base(0, 2) == '0'
    assert int(to_base(123456, 7), 7) == 123456


test_ex_2_9()
print('2.9 passed')

### 2.10 · Rounding half up  `Practice`

**Task.** `round(2.5)` is 2 in Python (banker's rounding). Write `round_half_up(x)` for non-negative floats that returns 3 for 2.5, using `math.floor`.

**Example.** `round(2.5)` -> `2`; `round_half_up(2.5)` -> `3`; `round_half_up(2.4)` -> `2`

**Big O.** O(1).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-2-10) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-2-10)

In [ ]:
# 2.10 Rounding half up
def round_half_up(x):
    raise NotImplementedError

In [ ]:
def test_ex_2_10():
    assert round(2.5) == 2
    assert round_half_up(2.5) == 3
    assert round_half_up(2.4) == 2


test_ex_2_10()
print('2.10 passed')